In [13]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

# Rail Corrugation — Model Training & Validation

Goal: classify each rail sample as:
- Normal
- Side I
- Side II

Official evaluation metric: Macro F1.

In [14]:

X_PATH = "../data/processed/X_train.npy"
Y_PATH = "../data/processed/y_train.npy"

In [15]:
X = np.load(X_PATH)
y = np.load(Y_PATH)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (272, 27)
y shape: (272,)


In [16]:
classes, counts = np.unique(y, return_counts=True)

print("Class distribution:")

for cls, count in zip(classes, counts):
    print(f"{cls}: {count}")

Class distribution:
0: 234
1: 14
2: 24


In [17]:
print("NaNs:", np.isnan(X).sum())
print("Infinities:", np.isinf(X).sum())

print("Minimum value:", np.min(X))
print("Maximum value:", np.max(X))

NaNs: 0
Infinities: 0
Minimum value: -0.57640725
Maximum value: 1313.0


In [18]:
print("First sample:")
print(X[0])

print("\nFirst 10 labels:")
print(y[:10])

First sample:
[ 1.1800000e+02  1.7966239e-02 -1.1025257e-01  1.8095298e-01
  3.8404119e-01  6.9236755e-01  1.4892578e+00  1.6298921e-01
  3.7951669e-01  1.6298676e-01  3.1079319e-01  6.3476562e-01
  1.3305664e+00  1.5303993e-01  3.0951813e-01  1.1593105e+00
  3.0295699e+00  6.3613892e+00  1.7529297e+01  9.0824872e-01
  2.5378833e+00  1.2695630e+00  2.5667708e+00  7.2731018e+00
  1.2377930e+01  9.9114925e-01  2.2669911e+00]

First 10 labels:
[0 2 0 0 0 0 0 0 0 0]


## Baseline Model — Random Forest

Because the dataset is highly imbalanced, model performance will be evaluated
using Macro F1 rather than accuracy.

A class-balanced Random Forest is used as the initial baseline.

In [28]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [30]:
from sklearn.model_selection import cross_val_score

rf_scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="f1_macro"
)

print("Macro F1 for each fold:")
print(rf_scores)

print("\nMean Macro F1:", rf_scores.mean())
print("Standard deviation:", rf_scores.std())

Macro F1 for each fold:
[0.53472222 0.56649832 0.96611274 0.65635739 0.75587076]

Mean Macro F1: 0.6959122863448763
Standard deviation: 0.1554979785918675


In [31]:
from sklearn.model_selection import cross_val_predict

rf_pred = cross_val_predict(
    rf,
    X,
    y,
    cv=cv
)

In [32]:
from sklearn.metrics import classification_report

print(classification_report(y, rf_pred))

              precision    recall  f1-score   support

           0       0.94      0.99      0.96       234
           1       0.50      0.21      0.30        14
           2       0.90      0.79      0.84        24

    accuracy                           0.93       272
   macro avg       0.78      0.66      0.70       272
weighted avg       0.92      0.93      0.92       272



In [33]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, rf_pred)

print(cm)

[[231   2   1]
 [ 10   3   1]
 [  4   1  19]]


Main weakness in class 1, which is pulling the Macro F1 down, Out of the 14 class-1 samples, you're only correctly detecting 3. Ten are being mistaken for class 0.

Can another model separate class 1 from class 0 better using the same 27 features?

In [34]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="rbf",
        C=1.0,
        class_weight="balanced"
    )
)

In [35]:
svm_scores = cross_val_score(
    svm,
    X,
    y,
    cv=cv,
    scoring="f1_macro"
)

print("SVM Macro F1 for each fold:")
print(svm_scores)

print("\nMean Macro F1:", svm_scores.mean())
print("Standard deviation:", svm_scores.std())

SVM Macro F1 for each fold:
[0.73639456 0.60090752 0.75505051 0.80772947 0.71101056]

Mean Macro F1: 0.7222185217977791
Standard deviation: 0.0684587450759531


In [36]:
svm_pred = cross_val_predict(
    svm,
    X,
    y,
    cv=cv
)

print(classification_report(y, svm_pred))

print("Confusion Matrix:")
print(confusion_matrix(y, svm_pred))

              precision    recall  f1-score   support

           0       0.97      0.90      0.93       234
           1       0.33      0.64      0.44        14
           2       0.70      0.79      0.75        24

    accuracy                           0.88       272
   macro avg       0.67      0.78      0.71       272
weighted avg       0.91      0.88      0.89       272

Confusion Matrix:
[[211  15   8]
 [  5   9   0]
 [  2   3  19]]


The important improvement is class 1. Random Forest found only 3/14 class-1 samples, while SVM found 9/14

The price is that SVM generates more false positives: 15 class-0 samples get classified as class 1. But because the competition cares about Macro F1 rather than accuracy, improving the minority classes can matter substantially.

In [37]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(
    svm,
    param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

grid.fit(X, y)

print("Best parameters:", grid.best_params_)
print("Best CV Macro F1:", grid.best_score_)

Best parameters: {'svc__C': 100, 'svc__gamma': 0.01}
Best CV Macro F1: 0.7861193957419078


In [38]:
best_svm = grid.best_estimator_

best_pred = cross_val_predict(
    best_svm,
    X,
    y,
    cv=cv
)

print(classification_report(y, best_pred))
print(confusion_matrix(y, best_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.97       234
           1       0.62      0.71      0.67        14
           2       0.80      0.67      0.73        24

    accuracy                           0.93       272
   macro avg       0.80      0.78      0.79       272
weighted avg       0.93      0.93      0.93       272

[[227   3   4]
 [  4  10   0]
 [  5   3  16]]


Tuned SVM is amazing guyss

## Final Model

The tuned SVM achieved the best cross-validation Macro F1 and is selected
as the final model.

Parameters:
- Kernel: RBF
- C: 100
- Gamma: 0.01
- Class weight: balanced

In [40]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

final_model = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="rbf",
        C=100,
        gamma=0.01,
        class_weight="balanced"
    )
)

# Train final model using ALL 272 training samples
final_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",100
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.01


In [41]:
import joblib

MODEL_PATH = "../models/rail_corrugation_svm.pkl"

joblib.dump(final_model, MODEL_PATH)

print("Saved final model to:", MODEL_PATH)

Saved final model to: ../models/rail_corrugation_svm.pkl
